In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease
import os

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

In [ ]:
# runs in Chameleon Jupyter environment
import datetime

username = os.getenv('USER')
l = lease.Lease(
    f"serve_model_{username}_proj13",
    duration=datetime.timedelta(hours=8)
)
l.add_node_reservation(node_type="compute_skylake", count=1)
l.submit(idempotent=True)
l.wait()
l.show()

In [ ]:
# runs in Chameleon Jupyter environment
s = server.Server(
    f"node-serve-model-{username}-proj13",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04",
)
s.submit(idempotent=True)

In [ ]:
# runs in Chameleon Jupyter environment
s.associate_floating_ip()
s.refresh()
s.check_connectivity()

In [ ]:
# runs in Chameleon Jupyter environment
s.refresh()
s.show(type="widget")

In [ ]:
# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("sudo apt-get install -y docker-compose-plugin")

In [ ]:
# runs in Chameleon Jupyter environment
REPO_URL = "https://github.com/<ORG>/SmartQueue.git"
s.execute(f"git clone {REPO_URL} ~/smartqueue")

In [ ]:
# runs in Chameleon Jupyter environment
s.execute("pip install torch --index-url https://download.pytorch.org/whl/cpu -q")
s.execute("cd ~/smartqueue/serving && python models/create_model.py")

Open an SSH session. From your local terminal, run:

```
ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```

where A.B.C.D is the floating IP shown in `s.show()` above.

In [ ]:
# runs on node-serve-model
# docker compose -f ~/smartqueue/serving/docker/docker-compose-model.yaml up --build -d

In [ ]:
# runs on node-serve-model
# docker exec jupyter jupyter server list

Paste the token URL into a browser tab, replacing `127.0.0.1` with the floating IP of your instance.

Then open notebooks 1–5 from the `work/` directory.

In [ ]:
# runs in Chameleon Jupyter environment
# delete instance when done to stop billing
s.delete()